## Split The Dataset into model classification structure gagawing cls yung obj det structure

In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np

def read_yolo_label(label_path):
    """Read YOLO label file and return class indices."""
    with open(label_path, 'r') as f:
        lines = f.readlines()
    class_indices = set()
    for line in lines:
        if line.strip():
            class_idx = int(line.split()[0])
            class_indices.add(class_idx)
    return class_indices

def separate_classes(base_dir, split_type):
    """Separate images and labels by class for a given split (train/val)."""
    images_dir = Path(base_dir) / split_type / 'images'
    labels_dir = Path(base_dir) / split_type / 'labels'
    
    # Create dictionary to store class information
    class_files = {}
    
    # Process each label file
    for label_file in labels_dir.glob('*.txt'):
        image_name = label_file.stem
        image_file = images_dir / f"{image_name}.jpg"  # Assuming jpg format, modify if needed
        
        # Skip if image doesn't exist
        if not image_file.exists():
            print(f"Warning: No matching image for {label_file}")
            continue
            
        # Get class indices from label file
        class_indices = read_yolo_label(label_file)
        
        # Add files to corresponding classes
        for class_idx in class_indices:
            if class_idx not in class_files:
                class_files[class_idx] = []
            class_files[class_idx].append((image_file, label_file))
    
    # Create directories and copy files
    for class_idx in class_files:
        # Create class directories
        class_dir = Path(base_dir) / split_type / f'class{class_idx}'
        class_images_dir = class_dir / 'images'
        class_labels_dir = class_dir / 'labels'
        
        os.makedirs(class_images_dir, exist_ok=True)
        os.makedirs(class_labels_dir, exist_ok=True)
        
        # Copy files
        for image_file, label_file in class_files[class_idx]:
            shutil.copy2(image_file, class_images_dir / image_file.name)
            shutil.copy2(label_file, class_labels_dir / label_file.name)
            
        print(f"{split_type}/class{class_idx}: {len(class_files[class_idx])} files")

def main():
    base_dir = 'F:\dataset - Copy'  # Change this to your dataset path
    
    # Process both train and val splits
    for split_type in ['train', 'val']:
        print(f"\nProcessing {split_type} split:")
        separate_classes(base_dir, split_type)

if __name__ == "__main__":
    main()

In [ ]:
import os
import shutil
import random

# --- Configuration ---
source_images = r"F:\SNAPFOLIA\OBJ_DET_DS_40 haha\images"
source_labels = r"F:\SNAPFOLIA\OBJ_DET_DS_40 haha\labels"
output_folder = r"F:\SNAPFOLIA\OBJ_REAL_FOLDER"
train_ratio = 0.8  # 80% for training

# --- Destination paths ---
train_images = os.path.join(output_folder, "train", "images")
train_labels = os.path.join(output_folder, "train", "labels")
test_images = os.path.join(output_folder, "test", "images")
test_labels = os.path.join(output_folder, "test", "labels")

# --- Create target directories ---
for path in [train_images, train_labels, test_images, test_labels]:
    os.makedirs(path, exist_ok=True)

# --- Get list of images ---
image_files = [f for f in os.listdir(source_images) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(image_files)

# --- Split dataset ---
split_index = int(len(image_files) * train_ratio)
train_files = image_files[:split_index]
test_files = image_files[split_index:]

# --- Move files ---
def move_files(file_list, img_dest, lbl_dest):
    for img_file in file_list:
        label_file = os.path.splitext(img_file)[0] + ".txt"

        # Move image
        shutil.copy(os.path.join(source_images, img_file), os.path.join(img_dest, img_file))

        # Move label if exists
        src_label_path = os.path.join(source_labels, label_file)
        if os.path.exists(src_label_path):
            shutil.move(src_label_path, os.path.join(lbl_dest, label_file))
        else:
            print(f"[!] Warning: Label file missing for {img_file}")

move_files(train_files, train_images, train_labels)
move_files(test_files, test_images, test_labels)

print("✅ Dataset successfully reorganized!")


✅ Dataset successfully reorganized!
